# Package workflow

Install `alzheimers-rag` into this kernel first. Run selected cells deliberately; model and database stages require their optional dependencies. See the repository README for the CLI.


In [ ]:
from pathlib import Path
from alzheimers_rag.config import DataPaths

# Works when launched from the repository root or notebooks directory.
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
paths = DataPaths(root / "data")


## Clean and inspect
Uses the existing local PMC corpus.


In [ ]:
from alzheimers_rag.pipeline import clean_corpus
from alzheimers_rag.io import read_jsonl
from alzheimers_rag.analysis import compute_corpus_metrics

count = clean_corpus(paths.raw, paths.cleaned)
metrics = compute_corpus_metrics(read_jsonl(paths.cleaned))
count, metrics["canonical_counts"]


## Chunk and embed
Run `alzheimers-rag setup-nltk` once before chunking. These cells download MedCPT model assets when needed.


In [ ]:
from alzheimers_rag.pipeline import chunk_corpus

chunk_corpus(paths.cleaned, paths.parents, paths.children)


In [ ]:
from alzheimers_rag.embeddings import embed_children

embed_children(paths.children, paths.embeddings, paths.id_map)


## Index and retrieve
Start Weaviate before running these cells.


In [ ]:
from alzheimers_rag.storage.weaviate import connect, index_children

with connect() as client:
    count = index_children(client, paths.children, paths.embeddings, paths.id_map)
count


In [ ]:
from alzheimers_rag.embeddings import MedCPTEncoder
from alzheimers_rag.retrieval.search import HybridRetriever
from alzheimers_rag.generation import build_prompt

question = "What causes neurofibrillary tangles?"
with connect() as client:
    retriever = HybridRetriever(client.collections.get("AlzheimersChunk"), MedCPTEncoder.for_queries())
    hits = retriever.search(question)
parents = {p["parent_id"]: p for p in read_jsonl(paths.parents)}
result = build_prompt(question, hits, parents=parents)
print(result["prompt"])


## Evaluate labeled retrieval pools
These are retrieval metrics over the existing judged pool.


In [ ]:
from alzheimers_rag.evaluation import load_judgments, compute_metrics

compute_metrics(load_judgments(root / "data/evaluation/judgments.csv"))
